# 🧠 EduSense RAG System
**King Khalid University — Graduation Project 2025**

## Pipeline
```
Upload PDFs → ChromaDB
     ↓
Student confused → Whisper transcribes lecture
     ↓
RAG retrieves relevant textbook content
     ↓
Claude generates personalized .ipynb notebook
     ↓
Student downloads notebook
```

## 📦 1. Install Dependencies

In [ ]:
!pip install chromadb sentence-transformers anthropic openai-whisper PyMuPDF -q
print('✅ Dependencies installed')

## 🔧 2. Setup

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

# Upload edusense_rag.py to Colab first
# Then import it
import sys
sys.path.append('/content')

# ⚠️ PUT YOUR API KEY HERE
ANTHROPIC_API_KEY = 'YOUR_KEY_HERE'  # get from console.anthropic.com

KB_DIR     = '/content/drive/MyDrive/edusense_kb'
OUTPUT_DIR = '/content/drive/MyDrive/edusense_notebooks'

print('✅ Setup done')

## 📚 3. Initialize RAG System

In [ ]:
from edusense_rag import EduSenseRAG

rag = EduSenseRAG(
    anthropic_api_key = ANTHROPIC_API_KEY,
    kb_dir            = KB_DIR,
    output_dir        = OUTPUT_DIR,
    whisper_model     = 'base'
)

## 📄 4. Upload Textbooks / Slides
Run this once to index your PDFs. After that they're saved in ChromaDB.

In [ ]:
# Option A: Upload from your computer
uploaded = files.upload()  # select your PDF files
pdf_paths = [f'/content/{name}' for name in uploaded.keys()]

# Option B: Load from Drive (if already uploaded)
# import glob
# pdf_paths = glob.glob('/content/drive/MyDrive/textbooks/*.pdf')

# Index them
rag.upload_textbooks(pdf_paths)
rag.knowledge_base_stats()

## 🧪 5. Test with Sample Transcript
Test the full pipeline without needing audio

In [ ]:
# Simulated emotion state (replace with real model output)
emotion_state = {
    'engagement':  {'positive': False, 'confidence': 0.32},
    'boredom':     {'positive': True,  'confidence': 0.74},
    'confusion':   {'positive': True,  'confidence': 0.81},
    'frustration': {'positive': False, 'confidence': 0.25},
}

# Sample lecture transcript
sample_transcript = """
So binary search trees, right, the key property is that for any node,
everything in the left subtree is smaller, everything in the right is larger.
The search operation takes O of log n time on average. But in the worst case,
if the tree is unbalanced, it degrades to O of n. That's why we have
balanced BSTs like AVL trees and red-black trees.
"""

# Generate notebook
notebook_path = rag.process_with_text(
    transcript    = sample_transcript,
    emotion_state = emotion_state
)

print(f'\n📓 Notebook generated: {notebook_path}')

## 🎙️ 6. Full Pipeline with Audio File

In [ ]:
# Upload audio recording
print('Upload your lecture audio file...')
audio_upload = files.upload()
audio_path   = f'/content/{list(audio_upload.keys())[0]}'

# Process
notebook_path = rag.process_dissatisfaction(
    audio_path    = audio_path,
    emotion_state = emotion_state,
    last_n_secs   = 60
)

print(f'\n📓 Notebook generated: {notebook_path}')

## 💾 7. Download Generated Notebook

In [ ]:
# Download the latest generated notebook
files.download(notebook_path)
print(f'✅ Downloaded: {notebook_path}')

## 🔗 8. Connect with EduSense Model
Integration with your trained emotion detection model

In [ ]:
# This cell shows how to connect your trained model with the RAG system
# Run after loading your EduSense model

def run_edusense_session(
    edusense_model,       # your trained model
    feature_extractor,    # AffectNet-ViT extractor
    rag_system,           # EduSenseRAG instance
    webcam_frames,        # list of 30 frames
    audio_path,           # lecture recording
    device='cuda'
):
    import torch
    import torch.nn.functional as F
    from collections import deque

    # Step 1: Extract embeddings
    embs = feature_extractor.extract(webcam_frames)  # (30, 768)
    seq  = torch.tensor(embs, dtype=torch.float32).unsqueeze(0).to(device)
    seq  = F.normalize(seq, p=2, dim=2)

    # Step 2: Get emotion predictions
    edusense_model.eval()
    with torch.no_grad():
        logits = edusense_model(seq)

    emotion_names = ['engagement', 'boredom', 'confusion', 'frustration']
    emotion_state = {}
    for i, name in enumerate(emotion_names):
        prob = torch.sigmoid(logits[i]).item()
        emotion_state[name] = {'positive': prob > 0.5, 'confidence': prob}

    print('Emotions detected:')
    for name, v in emotion_state.items():
        status = '✅' if v['positive'] else '❌'
        print(f'  {name}: {status} ({v["confidence"]:.2f})')

    # Step 3: Check dissatisfaction
    not_engaged = not emotion_state['engagement']['positive']
    negative    = any(emotion_state[e]['positive']
                     for e in ['boredom', 'confusion', 'frustration'])

    if not_engaged and negative:
        print('\n⚠️ Student dissatisfied — generating notebook...')
        notebook = rag_system.process_dissatisfaction(
            audio_path    = audio_path,
            emotion_state = emotion_state
        )
        return notebook
    else:
        print('\n✅ Student appears engaged')
        return None

print('✅ Integration function ready')
print('Call run_edusense_session() to trigger the full pipeline')